In [10]:
import os
import json
import math

ROOT = os.path.join('..', 'results', 'table2')

methods = ['NONE', 'SDRF', 'FOSR', 'LASER']

# (subdir suffix, pretty name, metric key, maximize?)
tasks = [
    ('func',   'Peptides-func',   'ap',  True),   # AP, higher is better
    ('struct', 'Peptides-struct', 'mae', False),  # MAE, lower is better
]


def load_stats(path):
    with open(path) as f:
        content = f.read().strip()

    # Try normal JSON first
    try:
        data = json.loads(content)
    except json.JSONDecodeError:
        # Fallback: JSON Lines (one JSON object per line)
        rows = []
        for line in content.splitlines():
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
        data = rows

    # Normalize formats
    if isinstance(data, list):
        # list of dicts
        return data
    elif isinstance(data, dict):
        # dict of lists -> convert to list of dicts
        keys = list(data.keys())
        n = len(data[keys[0]])
        rows = []
        for i in range(n):
            row = {k: data[k][i] for k in keys}
            rows.append(row)
        return rows
    else:
        raise ValueError(f"Unknown stats format in {path}")


def best_epoch(stats, metric, maximize):
    if maximize:
        best = max(stats, key=lambda r: r[metric])
    else:
        best = min(stats, key=lambda r: r[metric])
    return best["epoch"]


def mean_std(xs):
    n = len(xs)
    m = sum(xs) / n
    var = sum((x - m) ** 2 for x in xs) / n
    return m, math.sqrt(var)


def get_method_results(method, task_suffix, metric, maximize):
    vals = []
    best_epochs = []
    for seed in range(4):
        base = os.path.join(ROOT, f'{method}_{task_suffix}', str(seed))

        val_path = os.path.join(base, 'val',  'stats.json')
        test_path = os.path.join(base, 'test', 'stats.json')

        val_stats = load_stats(val_path)
        test_stats = load_stats(test_path)

        be = best_epoch(val_stats, metric, maximize)
        best_epochs.append(be)

        # find matching epoch in test stats
        test_row = next(r for r in test_stats if r['epoch'] == be)
        vals.append(test_row[metric])

    return mean_std(vals), vals, best_epochs


for task_suffix, task_name, metric, maximize in tasks:
    print(f'=== {task_name} ({metric}) ===')
    for method in methods:
        (m, s), per_seed, best_epochs = get_method_results(
            method, task_suffix, metric, maximize
        )
        seeds_str = ", ".join(f"{v:.4f}" for v in per_seed)
        epochs_str = ", ".join(str(e) for e in best_epochs)
        print(f'{(method if method != "NONE" else "None"):6s}: {m:.4f} ± {s:.4f}      '
              f'(seeds: {seeds_str}; epochs: {epochs_str})')
    print()

=== Peptides-func (ap) ===
None  : 0.5941 ± 0.0014      (seeds: 0.5918, 0.5941, 0.5952, 0.5952; epochs: 477, 295, 467, 227)
SDRF  : 0.5983 ± 0.0030      (seeds: 0.5984, 0.6009, 0.5933, 0.6006; epochs: 495, 301, 497, 287)
FOSR  : 0.4698 ± 0.0055      (seeds: 0.4689, 0.4626, 0.4696, 0.4781; epochs: 50, 91, 71, 77)
LASER : 0.6521 ± 0.0069      (seeds: 0.6482, 0.6634, 0.6512, 0.6453; epochs: 137, 286, 143, 154)

=== Peptides-struct (mae) ===
None  : 0.3477 ± 0.0008      (seeds: 0.3491, 0.3471, 0.3474, 0.3472; epochs: 245, 193, 214, 196)
SDRF  : 0.3462 ± 0.0012      (seeds: 0.3480, 0.3451, 0.3464, 0.3453; epochs: 153, 150, 145, 182)
FOSR  : 0.3095 ± 0.0013      (seeds: 0.3092, 0.3083, 0.3087, 0.3117; epochs: 73, 103, 89, 70)
LASER : 0.3128 ± 0.0019      (seeds: 0.3129, 0.3096, 0.3140, 0.3145; epochs: 96, 78, 136, 109)

